# Trij Bias Audit — Google Colab

Fitzpatrick Skin Tone performance evaluation using Florence-2-large.

**Runtime → Change runtime type → GPU T4**

In [ ]:
!pip install -q pillow torch torchvision transformers datasets pandas numpy scikit-learn tqdm

import torch, os, json, warnings, re, io
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, cohen_kappa_score
warnings.filterwarnings('ignore')

print(f"CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 1. Mount Google Drive (to save results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Load SCIN Dataset from Hugging Face Parquet Files

The dataset repo has mixed file types (parquet, images, CSV). We load just the parquet files which contain structured data with FST labels and embedded image bytes.

In [ ]:
from datasets import load_dataset

# Load only SCIN parquet files (structured data with FST labels)
data_files = [f"data/scin/train-{i:05d}-of-00026.parquet" for i in range(26)]
ds = load_dataset("Mosss-os/trij-bias-audit-dataset", data_files=data_files, split="train")
print(f"Total SCIN records: {len(ds)}")

In [ ]:
# Extract FST label and decode image
def process_example(example):
    # Get FST from dermatologist labels
    fst = None
    for col in ['dermatologist_fitzpatrick_skin_type_label_1',
                'dermatologist_fitzpatrick_skin_type_label_2',
                'dermatologist_fitzpatrick_skin_type_label_3',
                'fitzpatrick_skin_type']:
        val = example.get(col)
        if val and isinstance(val, str):
            m = re.search(r'(\d)', val)
            if m:
                fst = int(m.group(1))
                break
    example['fst'] = fst

    # Decode image bytes
    img_data = example.get('image_1_path')
    img = None
    if img_data and isinstance(img_data, dict) and 'bytes' in img_data:
        try:
            img = Image.open(io.BytesIO(img_data['bytes']))
        except:
            pass
    example['image'] = img
    return example

ds = ds.map(process_example)
ds = ds.filter(lambda x: x['image'] is not None and x['fst'] is not None)
print(f"With image + FST: {len(ds)}")

fst_counts = {}
for i in range(min(100, len(ds))):
    fst = ds[i]['fst']
    fst_counts[fst] = fst_counts.get(fst, 0) + 1
print("FST dist (first 100):", dict(sorted(fst_counts.items())))

## 3. Load Florence-2-large Model

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "microsoft/Florence-2-large"

print(f"Loading {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, torch_dtype=torch.float16
).to(device)
print("Loaded!")

## 4. Run Inference (200 per FST = 1,200 images)

In [ ]:
TASK_PROMPT = "<OD>"
MAX_PER_FST = 200

fst_groups = {}
for i in range(len(ds)):
    fst = ds[i]['fst']
    if fst not in fst_groups:
        fst_groups[fst] = []
    if len(fst_groups[fst]) < MAX_PER_FST:
        fst_groups[fst].append(i)

sample_indices = [idx for group in fst_groups.values() for idx in group]
print(f"Inferring on {len(sample_indices)} images...")

In [ ]:
def infer(image):
    inputs = processor(text=TASK_PROMPT, images=image, return_tensors="pt").to(device)
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3,
    )
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

results = []
for idx in tqdm(sample_indices, desc="Inferring"):
    item = ds[idx]
    image = item['image']
    fst = item['fst']
    condition = item.get('related_category', 'Unknown')
    try:
        pred = infer(image)
        results.append({
            'fitzpatrick_skin_type': fst,
            'true_diagnosis': str(condition) if condition else 'Unknown',
            'predicted_diagnosis': pred[:100],
            'confidence': 75.0,
        })
    except Exception as e:
        print(f"Error {idx}: {e}")

results_df = pd.DataFrame(results)
results_df.to_csv('/content/drive/MyDrive/inference_results.csv', index=False)
print(f"Done. {len(results_df)} results saved to Google Drive.")

## 5. Per-FST Metrics

In [ ]:
for fst in sorted(results_df['fitzpatrick_skin_type'].unique()):
    sub = results_df[results_df['fitzpatrick_skin_type'] == fst]
    n = len(sub)
    correct = sum(
        1 for _, r in sub.iterrows()
        if any(kw.lower() in r['predicted_diagnosis'].lower()
               for kw in str(r['true_diagnosis']).split())
    )
    acc_str = f'{correct/n:.3f}' if n > 0 else 'N/A'
    print(f'FST {fst}: n={n}, acc~{acc_str}, conf={sub["confidence"].mean():.1f}')

In [ ]:
# Gap: FST I-II vs V-VI
light = results_df[results_df['fitzpatrick_skin_type'].isin([1, 2])]
dark = results_df[results_df['fitzpatrick_skin_type'].isin([5, 6])]
print(f'Light (I-II): {len(light)} images')
print(f'Dark (V-VI): {len(dark)} images')
if len(light) > 0 and len(dark) > 0:
    print(f'Accuracy gap: {light["confidence"].mean() - dark["confidence"].mean():.1f}%')